In [0]:
%pip install -q torch transformers accelerate cloudpickle pandas mlflow
dbutils.library.restartPython()

In [0]:
import mlflow
import pandas as pd
from datetime import datetime

catalog = "workspace"
schema = "financial_sentiment"
table_prefix = f"{catalog}.{schema}"

mlflow.set_registry_uri("databricks-uc")

MODELS_TO_TEST = [
    {
        "model_key": "finbert",
        "model_name": f"{catalog}.{schema}.financial_sentiment_finbert",
        "version": 1
    },
    {
        "model_key": "distilroberta_financial",
        "model_name": f"{catalog}.{schema}.financial_sentiment_distilroberta_financial",
        "version": 1
    }
]

texts = pd.DataFrame({
    "text_clean": [
        "the company reported strong revenue growth and better margins",
        "the bank faces losses due to market uncertainty",
        "the quarterly report was in line with expectations"
    ]
})

all_predictions = []

for model_info in MODELS_TO_TEST:
    model_uri = f"models:/{model_info['model_name']}/{model_info['version']}"
    model = mlflow.pyfunc.load_model(model_uri)

    pred_df = model.predict(texts)

    output = texts.copy()
    output["prediction"] = pred_df["prediction"]
    output["score"] = pred_df["score"]
    output["model_key"] = model_info["model_key"]
    output["model_name"] = model_info["model_name"]
    output["model_version"] = model_info["version"]
    output["prediction_timestamp"] = datetime.now()

    all_predictions.append(output)

final_predictions = pd.concat(all_predictions, ignore_index=True)

display(final_predictions)

target_table = f"{table_prefix}.gold_financial_sentiment_predictions"

existing_cols = [
    row.col_name
    for row in spark.sql(f"DESCRIBE TABLE {target_table}").collect()
    if row.col_name and not row.col_name.startswith("#")
]

print(existing_cols)

if "score" not in existing_cols:
    spark.sql(f"""
        ALTER TABLE {target_table}
        ADD COLUMNS (score DOUBLE)
    """)

if "model_key" not in existing_cols:
    spark.sql(f"""
        ALTER TABLE {target_table}
        ADD COLUMNS (model_key STRING)
    """)

print("Esquema actualizado")

spark.createDataFrame(final_predictions).write.mode("append").saveAsTable(
    f"{table_prefix}.gold_financial_sentiment_predictions"
)